# Импорт библиотек

In [2]:
import pandas as pd
pd.set_option('display.max_columns', None)

In [3]:
df = pd.read_csv('krisha.csv')

In [10]:
df.shape

(19876, 31)

In [4]:
df["address"] = (
    df["title"]
    .str.extract(r"этаж,\s*(.*)$")[0]
    .str.strip()
)

In [5]:
df = df.drop(columns='Unnamed: 0')

# EDA

Очень много фичей на данный момент, которые имеют очень слабый сигнал

In [6]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 19876 entries, 0 to 19875
Data columns (total 31 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   id                   19876 non-null  int64  
 1   url                  19876 non-null  object 
 2   title                19876 non-null  object 
 3   price                19876 non-null  int64  
 4   price_raw            19876 non-null  object 
 5   price_per_m2         19876 non-null  float64
 6   rooms                19876 non-null  int64  
 7   area                 19876 non-null  float64
 8   floor                18114 non-null  float64
 9   total_floors         17817 non-null  float64
 10  location_raw         19876 non-null  object 
 11  city                 19876 non-null  object 
 12  district             19441 non-null  object 
 13  residential_complex  10769 non-null  object 
 14  house_type           18299 non-null  object 
 15  year_built           19876 non-null 

Количество городов в объявлениях - 177

In [7]:
df['city'].nunique()

177

Самые инфoрмативные данные - это города Астана, Алматы и Шымкент. Остальные города имеют слишком мало данных, что приведет к шуму и проблемам для модели. 

Поэтому оставим только топ-3 города

In [15]:
df['city'].value_counts()

city
Алматы        6225
Астана        5860
Шымкент       1050
Актобе         591
Актау          574
              ... 
Дамса            1
Киевка           1
Бейнеу           1
Жамбыл           1
Фарфоровое       1
Name: count, Length: 177, dtype: int64

In [8]:
top_cities = ['Астана', 'Алматы', 'Шымкент']
df_copy = df[df['city'].isin(top_cities)]

In [17]:
df_copy.shape

(13135, 31)

Количество фичей 30, но из них лишь некоторая часть действительно может быть информативной. На первой итерации я отберу признаки в переменной important_cols, а также уберу все остальные, которые точно никакой информативности для модели не дадут

In [9]:
df_copy.columns

Index(['id', 'url', 'title', 'price', 'price_raw', 'price_per_m2', 'rooms',
       'area', 'floor', 'total_floors', 'location_raw', 'city', 'district',
       'residential_complex', 'house_type', 'year_built', 'condition',
       'bathroom', 'balcony', 'balcony_glazed', 'door', 'phone', 'internet',
       'parking', 'furniture', 'floor_covering', 'ceiling_height',
       'former_hostel', 'exchange_possible', 'description', 'address'],
      dtype='object')

In [10]:
important_cols = ['id', 'price', 'price_per_m2', 'rooms', 'area', 'floor', 'total_floors', 'city', 'district', 'residential_complex', 'house_type', 'year_built', 'condition', 'bathroom', 'parking', 'furniture', 'ceiling_height', 'address']

In [11]:
df_copy = df_copy[important_cols]

Долевое отношение пропущенных значений:

In [12]:
(df_copy.isna().sum().sort_values(ascending = False) * 100 / df_copy.shape[0]).round(2)

condition              48.33
furniture              42.14
parking                39.79
residential_complex    29.02
bathroom               25.51
total_floors           11.86
floor                  10.23
address                10.23
ceiling_height          9.96
house_type              5.93
district                3.06
id                      0.00
price                   0.00
area                    0.00
city                    0.00
price_per_m2            0.00
rooms                   0.00
year_built              0.00
dtype: float64

У нас имеются, как категориальные фичи, так и числовые. И те, и другие имеют пропуски. Особенно пропуски наблюдаются в условиях, мебелированности и паркинге

In [13]:
# Восстановление числовых признаков
pd.set_option('display.max_rows', None)
total_floors_median = df_copy.groupby(by=['city', 'district', 'residential_complex'])['total_floors'].median()
ceiling_height_median = df_copy.groupby(by=['city', 'district', 'residential_complex'])['ceiling_height'].median()

In [14]:
group_cols = ["city", "district", "residential_complex"]

median_floors = (
    df_copy.groupby(group_cols)["total_floors"]
    .median()
)

mask = df_copy["total_floors"].isna()

df_copy.loc[mask, "total_floors"] = (
    df_copy.loc[mask, group_cols]
    .merge(
        median_floors.rename("median_total_floors").reset_index(),
        on=group_cols,
        how="left"
    )["median_total_floors"]
    .values
)

In [15]:
group_cols = ["city", "district", "residential_complex"]

median_height = (
    df_copy.groupby(group_cols)["ceiling_height"]
    .median()
)

mask = df_copy["ceiling_height"].isna()

df_copy.loc[mask, "ceiling_height"] = (
    df_copy.loc[mask, group_cols]
    .merge(
        median_floors.rename("median_height").reset_index(),
        on=group_cols,
        how="left"
    )["median_height"]
    .values
)

Этажность была 11%, стало 4%

Высота потолков была 9, стало 8

In [16]:
(df_copy.isna().sum().sort_values(ascending = False) * 100 / df_copy.shape[0]).round(2)

condition              48.33
furniture              42.14
parking                39.79
residential_complex    29.02
bathroom               25.51
floor                  10.23
address                10.23
ceiling_height          8.30
house_type              5.93
total_floors            4.03
district                3.06
id                      0.00
price                   0.00
area                    0.00
city                    0.00
price_per_m2            0.00
rooms                   0.00
year_built              0.00
dtype: float64

Категориальные данные: city, district, residential_complex, house_type, condition, bathroom, parking, furniture

In [17]:
cats_cols = ['house_type', 'condition', 'bathroom', 'parking', 'furniture']
for col in cats_cols:
    print(f'{col}:', df_copy[col].unique())

house_type: ['монолитный' 'кирпичный' 'панельный' 'иной' nan]
condition: [nan 'свежий ремонт' 'не новый, но аккуратный ремонт' 'черновая отделка'
 'требует ремонта' 'свободная планировка']
bathroom: ['совмещенный' '2 с/у и более' nan 'раздельный' 'нет']
parking: ['паркинг' 'рядом охраняемая стоянка' nan 'гараж']
furniture: ['частично' 'полностью' nan 'без мебели']


In [18]:
pd.reset_option('display.max_rows')

In [19]:
df_copy

,id,price,price_per_m2,rooms,area,floor,total_floors,city,district,residential_complex,house_type,year_built,condition,bathroom,parking,furniture,ceiling_height,address
0,1012376197,31500000,484615.38,2,65.00,8.0,20.0,Астана,NaN,A-City,монолитный,2025,NaN,совмещенный,паркинг,частично,3.0,ул. Темирбека Жургенова 17
1,1012029993,73000000,811111.11,3,90.00,14.0,14.0,Астана,Есильский р-н,Turan Life,монолитный,2023,NaN,2 с/у и более,рядом охраняемая стоянка,полностью,2.7,ул. Абикена Бектурова
3,1012043735,77800000,1005167.96,3,77.40,10.0,12.0,Алматы,Алмалинский р-н,Shabyt,монолитный,2025,свежий ремонт,совмещенный,паркинг,частично,3.0,Просп. Сакена Сейфуллина
4,1006097639,120000000,1200000.00,3,100.00,6.0,16.0,Астана,Есильский р-н,Zangar,монолитный,2024,NaN,NaN,паркинг,NaN,3.0,Улы дала 37 — Ботанический сад/ Мега
6,690810657,88199712,669600.00,4,131.72,NaN,3.5,Шымкент,Енбекшинский р-н,Aulet Residence,кирпичный,2024,NaN,2 с/у и более,NaN,NaN,3.2,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
19864,1011837174,38700000,893764.43,1,43.30,3.0,9.0,Астана,Нура р-н,GreenLine.Asyl Mura Ualihanov,монолитный,2022,свежий ремонт,NaN,паркинг,NaN,3.0,Толе би 65
19865,1012447968,13500000,221311.48,1,61.00,11.0,16.0,Астана,Сарыарка р-н,KHAN RESIDENCE,монолитный,2026,NaN,NaN,NaN,NaN,2.8,Н. Тлендиева 52
19866,1011375461,24000000,446927.37,2,53.70,6.0,11.0,Шымкент,Туран р-н,NaN,монолитный,1988,"не новый, но аккуратный ремонт",совмещенный,NaN,NaN,NaN,желтоксан
19871,1011673732,35000000,466666.67,3,75.00,4.0,9.0,Алматы,Турксибский р-н,NaN,NaN,2026,NaN,NaN,NaN,NaN,NaN,"мкр Кайрат, ЖК Турар туркестан 115"


In [20]:
df_copy["address"].sample(20, random_state=42)

8772                             Ауэзова улица — Жамбыла
3972                                                 NaN
12448                                Жубан Молдагалиев 9
3282                                                 NaN
11230     мкр Самал-2 — По Мендикулова - ниже Аль-Фараби
10194                     Айнакол 64 — Жумабаева-Айнакөл
13635                                          Райымбека
5585                      мкр Шугыла, Сакен Жунисов 2/25
11966                          Азербаева 6 — Кошкарбаева
1105                                   Нажимеденова 34/2
11659                             Жумекен Нажимеденов 28
12454                                                NaN
3093                     Храпатый 14 — Анатолий Храпатый
12010                                           Туран 48
18264                                         Ардагерлер
2828                                            Туран 39
94                                        Байтурсынова 3
12826    Турар Рыскулов 16а — М

In [21]:
df_copy["address"].nunique()

9047

In [22]:
df_copy["address_clean"] = (
    df_copy["address"]
    .str.split("—")
    .str[0]
    .str.strip()
)

In [23]:
df_copy["address_clean"].nunique()

6863

In [24]:
(
    df_copy["address_clean"]
    .value_counts()
    .describe()
)

count    6863.000000
mean        1.718053
std         2.268530
min         1.000000
25%         1.000000
50%         1.000000
75%         2.000000
max        68.000000
Name: count, dtype: float64

feature engineering

In [25]:
# возраст дома
df_copy["house_age"] = 2026 - df_copy["year_built"]

# отношение этажа к этажности
df_copy["floor_ratio"] = (
    df_copy["floor"] / df_copy["total_floors"]
)

# первый этаж
df_copy["is_first_floor"] = (
    df_copy["floor"] == 1
).astype(int)

# последний этаж
df_copy["is_last_floor"] = (
    df_copy["floor"] == df_copy["total_floors"]
).astype(int)

# новостройка
df_copy["is_new_building"] = (
    df_copy["year_built"] >= 2020
).astype(int)

# высокий этаж
df_copy["is_high_floor"] = (
    df_copy["floor"] >= 15
).astype(int)

# малоэтажный дом
df_copy["is_low_rise"] = (
    df_copy["total_floors"] <= 5
).astype(int)

# большая квартира
df_copy["is_large_apartment"] = (
    df_copy["area"] >= 100
).astype(int)

In [26]:
df_copy

,id,price,price_per_m2,rooms,area,floor,total_floors,city,district,residential_complex,house_type,year_built,condition,bathroom,parking,furniture,ceiling_height,address,address_clean,house_age,floor_ratio,is_first_floor,is_last_floor,is_new_building,is_high_floor,is_low_rise,is_large_apartment
0,1012376197,31500000,484615.38,2,65.00,8.0,20.0,Астана,NaN,A-City,монолитный,2025,NaN,совмещенный,паркинг,частично,3.0,ул. Темирбека Жургенова 17,ул. Темирбека Жургенова 17,1,0.400000,0,0,1,0,0,0
1,1012029993,73000000,811111.11,3,90.00,14.0,14.0,Астана,Есильский р-н,Turan Life,монолитный,2023,NaN,2 с/у и более,рядом охраняемая стоянка,полностью,2.7,ул. Абикена Бектурова,ул. Абикена Бектурова,3,1.000000,0,1,1,0,0,0
3,1012043735,77800000,1005167.96,3,77.40,10.0,12.0,Алматы,Алмалинский р-н,Shabyt,монолитный,2025,свежий ремонт,совмещенный,паркинг,частично,3.0,Просп. Сакена Сейфуллина,Просп. Сакена Сейфуллина,1,0.833333,0,0,1,0,0,0
4,1006097639,120000000,1200000.00,3,100.00,6.0,16.0,Астана,Есильский р-н,Zangar,монолитный,2024,NaN,NaN,паркинг,NaN,3.0,Улы дала 37 — Ботанический сад/ Мега,Улы дала 37,2,0.375000,0,0,1,0,0,1
6,690810657,88199712,669600.00,4,131.72,NaN,3.5,Шымкент,Енбекшинский р-н,Aulet Residence,кирпичный,2024,NaN,2 с/у и более,NaN,NaN,3.2,NaN,NaN,2,NaN,0,0,1,0,1,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
19864,1011837174,38700000,893764.43,1,43.30,3.0,9.0,Астана,Нура р-н,GreenLine.Asyl Mura Ualihanov,монолитный,2022,свежий ремонт,NaN,паркинг,NaN,3.0,Толе би 65,Толе би 65,4,0.333333,0,0,1,0,0,0
19865,1012447968,13500000,221311.48,1,61.00,11.0,16.0,Астана,Сарыарка р-н,KHAN RESIDENCE,монолитный,2026,NaN,NaN,NaN,NaN,2.8,Н. Тлендиева 52,Н. Тлендиева 52,0,0.687500,0,0,1,0,0,0
19866,1011375461,24000000,446927.37,2,53.70,6.0,11.0,Шымкент,Туран р-н,NaN,монолитный,1988,"не новый, но аккуратный ремонт",совмещенный,NaN,NaN,NaN,желтоксан,желтоксан,38,0.545455,0,0,0,0,0,0
19871,1011673732,35000000,466666.67,3,75.00,4.0,9.0,Алматы,Турксибский р-н,NaN,NaN,2026,NaN,NaN,NaN,NaN,NaN,"мкр Кайрат, ЖК Турар туркестан 115","мкр Кайрат, ЖК Турар туркестан 115",0,0.444444,0,0,1,0,0,0


In [27]:
df_copy = df_copy.drop_duplicates(subset="id")

In [28]:
df_copy.info()

<class 'pandas.core.frame.DataFrame'>
Index: 12869 entries, 0 to 19874
Data columns (total 27 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   id                   12869 non-null  int64  
 1   price                12869 non-null  int64  
 2   price_per_m2         12869 non-null  float64
 3   rooms                12869 non-null  int64  
 4   area                 12869 non-null  float64
 5   floor                11600 non-null  float64
 6   total_floors         12366 non-null  float64
 7   city                 12869 non-null  object 
 8   district             12472 non-null  object 
 9   residential_complex  9105 non-null   object 
 10  house_type           12100 non-null  object 
 11  year_built           12869 non-null  int64  
 12  condition            6673 non-null   object 
 13  bathroom             9570 non-null   object 
 14  parking              7779 non-null   object 
 15  furniture            7476 non-null   obje

In [29]:
df_copy[["price", "price_per_m2", "area", "rooms", "floor", "total_floors"]].describe()

,price,price_per_m2,area,rooms,floor,total_floors
count,1.286900e+04,1.286900e+04,12869.000000,12869.000000,11600.000000,12366.000000
mean,5.084892e+07,7.117460e+05,68.695481,2.277877,6.005000,10.201116
std,4.559374e+07,2.541574e+05,36.499531,0.970273,4.137557,4.948721
min,4.000000e+06,9.676528e+04,10.000000,1.000000,1.000000,1.000000
25%,2.800000e+07,5.461538e+05,44.500000,2.000000,3.000000,6.000000
50%,3.900000e+07,6.713781e+05,60.500000,2.000000,5.000000,9.000000
75%,5.700000e+07,8.250000e+05,81.500000,3.000000,8.000000,12.000000
max,1.500000e+09,6.912442e+06,866.000000,17.000000,42.000000,114.000000


In [30]:
df_copy = df_copy[
    (df_copy["area"] <= 400) &
    (df_copy["rooms"] <= 10) &
    (df_copy["total_floors"] <= 50)
]

In [31]:
df_copy[["price", "price_per_m2", "area", "rooms", "floor", "total_floors"]].describe()

,price,price_per_m2,area,rooms,floor,total_floors
count,1.236200e+04,1.236200e+04,12362.000000,12362.000000,11528.000000,12362.000000
mean,4.928543e+07,7.077998e+05,67.554244,2.260718,6.006332,10.192687
std,4.052022e+07,2.456007e+05,33.548814,0.943545,4.134447,4.859177
min,4.000000e+06,9.676528e+04,10.000000,1.000000,1.000000,1.000000
25%,2.800000e+07,5.465116e+05,44.210000,2.000000,3.000000,6.000000
50%,3.900000e+07,6.710057e+05,60.000000,2.000000,5.000000,9.000000
75%,5.600000e+07,8.205128e+05,80.000000,3.000000,8.000000,12.000000
max,1.500000e+09,6.912442e+06,400.000000,8.000000,42.000000,43.000000


GEO

In [32]:
import numpy as np
df_geo = df_copy.copy()

In [33]:
df_geo = df_copy.copy()

df_geo["geocode_query"] = (
    df_geo["city"].fillna("") +
    ", " +
    df_geo["address_clean"].fillna("")
)

In [34]:
unique_queries = (
    df_geo["geocode_query"]
    .dropna()
    .drop_duplicates()
    .reset_index(drop=True)
)

len(unique_queries)

6858

In [35]:
from geopy.geocoders import Nominatim
from geopy.extra.rate_limiter import RateLimiter

geolocator = Nominatim(user_agent="krisha_price_prediction")

geocode = RateLimiter(
    geolocator.geocode,
    min_delay_seconds=1
)

test_queries = unique_queries

results = []

for query in test_queries:
    location = geocode(query)

    if location:
        results.append({
            "geocode_query": query,
            "lat": location.latitude,
            "lon": location.longitude
        })
    else:
        results.append({
            "geocode_query": query,
            "lat": None,
            "lon": None
        })

coords_df = pd.DataFrame(results)

In [36]:
coords_df.to_csv('coords_df.csv')

In [37]:
coords_df[coords_df["lat"].notna()].shape

(3880, 3)

In [38]:
df_geo = df_geo.merge(
    coords_df,
    on="geocode_query",
    how="left"
)

In [40]:
df_geo.to_csv('df_geo.csv')

Baseline

In [41]:
df_copy.columns

Index(['id', 'price', 'price_per_m2', 'rooms', 'area', 'floor', 'total_floors',
       'city', 'district', 'residential_complex', 'house_type', 'year_built',
       'condition', 'bathroom', 'parking', 'furniture', 'ceiling_height',
       'address', 'address_clean', 'house_age', 'floor_ratio',
       'is_first_floor', 'is_last_floor', 'is_new_building', 'is_high_floor',
       'is_low_rise', 'is_large_apartment'],
      dtype='object')

In [64]:
df_description = df_geo.merge(df[['description', 'id']], on='id', how='left').drop_duplicates(subset = 'id')

In [65]:
df_exp = df_description.copy()
df_exp["description"] = df_exp["description"].fillna("").astype(str)
df_exp["description_len"] = df_exp["description"].str.len()
df_almaty = df_copy[df_copy['city']=='Алматы']
threshold = df_almaty["price_per_m2"].quantile(0.99)
premium_ids = df_almaty[
    df_almaty["price_per_m2"] > threshold
]["id"]
df_exp = df_exp[
    ~df_exp["id"].isin(premium_ids)
].copy()

In [66]:
df_exp["coordinates_found"] = df_exp["lat"].notna().astype(int)

In [86]:
df_exp.to_csv('df_geo.csv')

In [67]:
# fallback 1: city + district
df_exp["lat"] = df_exp["lat"].fillna(
    df_exp.groupby(["city", "district"])["lat"].transform("median")
)

df_exp["lon"] = df_exp["lon"].fillna(
    df_exp.groupby(["city", "district"])["lon"].transform("median")
)

# fallback 2: city
df_exp["lat"] = df_exp["lat"].fillna(
    df_exp.groupby("city")["lat"].transform("median")
)

df_exp["lon"] = df_exp["lon"].fillna(
    df_exp.groupby("city")["lon"].transform("median")
)

In [68]:
df_exp["coordinates_found"].mean()

np.float64(0.6384915474642393)

In [85]:
target = "price_per_m2"
features = [
    'rooms',
    'area',
    'floor',
    'total_floors',

    'city',
    'district',
    'residential_complex',

    'house_type',

    'condition',
    'bathroom',

    'ceiling_height',

    'house_age',
    'floor_ratio',
    'is_first_floor',
    'is_last_floor',
    'description_len',

    'lat',
    'lon',
    'coordinates_found'
]

cat_features = [
    'city',
    'district',
    'residential_complex',
    'house_type',
    'condition',
    'bathroom'
]

for col in cat_features:
    df_exp[col] = df_exp[col].fillna("unknown").astype(str)
    
num_features = [col for col in features if col not in cat_features]

for col in num_features:
    df_exp[col] = df_exp[col].fillna(df_exp[col].median())


MLFLOW


In [70]:
import os
import json
import numpy as np
import pandas as pd
import mlflow
import mlflow.catboost

from catboost import CatBoostRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error


PROJECT_DIR = r"C:\Users\sandugash\Desktop\Prediction_pr_KZ"
MLRUNS_DIR = os.path.join(PROJECT_DIR, "mlruns")

mlflow.set_tracking_uri(f"file:///{MLRUNS_DIR.replace(os.sep, '/')}")
mlflow.set_experiment("krisha_kz_price_prediction")

Traceback (most recent call last):
  File "c:\Users\sandugash\AppData\Local\Programs\Python\Python39\lib\site-packages\mlflow\store\tracking\file_store.py", line 356, in search_experiments
    exp = self._get_experiment(exp_id, view_type)
  File "c:\Users\sandugash\AppData\Local\Programs\Python\Python39\lib\site-packages\mlflow\store\tracking\file_store.py", line 454, in _get_experiment
    meta = FileStore._read_yaml(experiment_dir, FileStore.META_DATA_FILE_NAME)
  File "c:\Users\sandugash\AppData\Local\Programs\Python\Python39\lib\site-packages\mlflow\store\tracking\file_store.py", line 1595, in _read_yaml
    return _read_helper(root, file_name, attempts_remaining=retries)
  File "c:\Users\sandugash\AppData\Local\Programs\Python\Python39\lib\site-packages\mlflow\store\tracking\file_store.py", line 1588, in _read_helper
    result = read_yaml(root, file_name)
  File "c:\Users\sandugash\AppData\Local\Programs\Python\Python39\lib\site-packages\mlflow\utils\yaml_utils.py", line 107, in 

<Experiment: artifact_location='file:///C:/Users/sandugash/Desktop/Prediction_pr_KZ/mlruns/189839478325921444', creation_time=1779902061858, experiment_id='189839478325921444', last_update_time=1779902061858, lifecycle_stage='active', name='krisha_kz_price_prediction', tags={}>

In [126]:
df = df_exp.copy()
X = df[features]
y = np.log1p(df['price_per_m2'])
print(X.columns)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.3,
    random_state=42
)

Index(['rooms', 'area', 'floor', 'total_floors', 'city', 'district',
       'residential_complex', 'house_type', 'condition', 'bathroom',
       'ceiling_height', 'house_age', 'floor_ratio', 'is_first_floor',
       'is_last_floor', 'description_len', 'lat', 'lon', 'coordinates_found'],
      dtype='object')


Best params: {'iterations': 1150, 'learning_rate': 0.07524709336659834, 'depth': 8, 'l2_leaf_reg': 6.319011040038374, 'bagging_temperature': 4.046353069828875}

In [84]:
params = {
    "iterations": 1000,
    "learning_rate": 0.05,
    "depth": 8,
    "loss_function": "RMSE",
    "eval_metric": "RMSE",
    "random_seed": 42,
    "verbose": 100
}

run_name = "price_per_m2_v20_without_district"

with mlflow.start_run(run_name=run_name):

    mlflow.log_params(params)

    mlflow.log_param("target", target)
    mlflow.log_param("model_type", "CatBoostRegressor")
    mlflow.log_param("feature_set", "price_per_m2_v20_without_district")
    mlflow.log_param("features", json.dumps(features, ensure_ascii=False))
    mlflow.log_param("cat_features", json.dumps(cat_features, ensure_ascii=False))

    mlflow.log_metric("train_rows", X_train.shape[0])
    mlflow.log_metric("test_rows", X_test.shape[0])
    mlflow.log_metric("n_features", len(features))

    model = CatBoostRegressor(**params)

    model.fit(
        X_train,
        y_train,
        cat_features=cat_features,
        eval_set=(X_test, y_test)
    )

    preds = model.predict(X_test)
    
    preds = np.expm1(preds)
    y_test = np.expm1(y_test)

    mae = mean_absolute_error(y_test, preds)
    rmse = np.sqrt(mean_squared_error(y_test, preds))
    mape = np.mean(
    np.abs((y_test - preds) / y_test) * 100)


    mlflow.log_metric("mae", mae)
    mlflow.log_metric("rmse", rmse)

    feature_importance = pd.DataFrame({
        "feature": features,
        "importance": model.feature_importances_
    }).sort_values("importance", ascending=False)

    os.makedirs("reports", exist_ok=True)

    fi_path = "reports/feature_importance_price_per_m2_v20_without_district.csv"
    feature_importance.to_csv(fi_path, index=False, encoding="utf-8-sig")

    mlflow.log_artifact(fi_path)

    mlflow.catboost.log_model(
        model,
        name="model",
        input_example=X_train.head(5)
    )

print("MAPE:", mape)
print("MAE:", mae)
print("RMSE:", rmse)

0:	learn: 0.2983225	test: 0.2959981	best: 0.2959981 (0)	total: 98.4ms	remaining: 1m 38s
100:	learn: 0.1692507	test: 0.1682193	best: 0.1682193 (100)	total: 6.27s	remaining: 55.8s
200:	learn: 0.1541234	test: 0.1619605	best: 0.1619605 (200)	total: 12.4s	remaining: 49.3s
300:	learn: 0.1442272	test: 0.1588467	best: 0.1588467 (300)	total: 18.3s	remaining: 42.5s
400:	learn: 0.1366861	test: 0.1572372	best: 0.1572354 (399)	total: 23.9s	remaining: 35.7s
500:	learn: 0.1309822	test: 0.1560075	best: 0.1559997 (497)	total: 29.5s	remaining: 29.4s
600:	learn: 0.1261599	test: 0.1552357	best: 0.1552357 (600)	total: 35.2s	remaining: 23.3s
700:	learn: 0.1218080	test: 0.1547148	best: 0.1547080 (688)	total: 40.9s	remaining: 17.5s
800:	learn: 0.1178923	test: 0.1542573	best: 0.1542573 (800)	total: 46.7s	remaining: 11.6s
900:	learn: 0.1134022	test: 0.1537936	best: 0.1537936 (900)	total: 53.1s	remaining: 5.84s
999:	learn: 0.1094033	test: 0.1534431	best: 0.1534431 (999)	total: 58.7s	remaining: 0us

bestTest = 0.

c:\Users\sandugash\AppData\Local\Programs\Python\Python39\lib\site-packages\mlflow\types\utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(

MAPE: 11.528207543352199
MAE: 83240.0499600794
RMSE: 120323.20561998867


In [89]:
pd.set_option('display.max_rows', None)
row = df_exp.iloc[0]
row

id                                                            1012376197
price                                                           31500000
price_per_m2                                                   484615.38
rooms                                                                  2
area                                                                65.0
floor                                                                8.0
total_floors                                                        20.0
city                                                              Астана
district                                                         unknown
residential_complex                                               A-City
house_type                                                    монолитный
year_built                                                          2025
condition                                                        unknown
bathroom                                           

In [91]:
pd.reset_option('display.max_rows')

In [98]:
for col in cat_features:
    print(f'{col}:', df_exp[col].unique())

city: ['Астана' 'Алматы' 'Шымкент']
district: ['unknown' 'Есильский р-н' 'Алмалинский р-н' 'Енбекшинский р-н'
 'Алатауский р-н' 'Нура р-н' 'Абайский р-н' 'Каратауский р-н'
 'Турксибский р-н' 'Сарайшык р-н' 'Бостандыкский р-н' 'Наурызбайский р-н'
 'Сарыарка р-н' 'р-н Байконур' 'Ауэзовский р-н' 'Алматы р-н'
 'Медеуский р-н' 'Аль-Фарабийский р-н' 'Жетысуский р-н' 'Туран р-н']
residential_complex: ['A-City' 'Turan Life' 'Shabyt' ... 'Inju Ishim' 'Улан' 'Park Avenue']
house_type: ['монолитный' 'кирпичный' 'панельный' 'иной' 'unknown']
condition: ['unknown' 'свежий ремонт' 'не новый, но аккуратный ремонт'
 'черновая отделка' 'требует ремонта' 'свободная планировка']
bathroom: ['совмещенный' '2 с/у и более' 'unknown' 'раздельный' 'нет']


In [131]:
df[features + ["price_per_m2", "price"]].to_csv('df_public.csv')

In [ ]:
df_exp[features].merge

,rooms,area,floor,total_floors,city,district,residential_complex,house_type,condition,bathroom,ceiling_height,house_age,floor_ratio,is_first_floor,is_last_floor,description_len,lat,lon,coordinates_found
0,2,65.00,8.0,20.0,Астана,unknown,A-City,монолитный,unknown,совмещенный,3.00,1,0.400000,0,0,220,51.120780,71.503799,1
1,3,90.00,14.0,14.0,Астана,Есильский р-н,Turan Life,монолитный,unknown,2 с/у и более,2.70,3,1.000000,0,1,1040,51.129084,71.400691,1
2,3,77.40,10.0,12.0,Алматы,Алмалинский р-н,Shabyt,монолитный,свежий ремонт,совмещенный,3.00,1,0.833333,0,0,575,43.250075,76.908222,0
3,3,100.00,6.0,16.0,Астана,Есильский р-н,Zangar,монолитный,unknown,unknown,3.00,2,0.375000,0,0,209,51.099164,71.407265,1
4,4,131.72,5.0,3.5,Шымкент,Енбекшинский р-н,Aulet Residence,кирпичный,unknown,2 с/у и более,3.20,2,0.600000,0,0,1686,42.314696,69.588328,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
12597,2,55.00,3.0,13.0,Алматы,Бостандыкский р-н,unknown,монолитный,unknown,2 с/у и более,2.85,7,0.230769,0,0,136,43.216412,76.890639,1
12598,1,43.30,3.0,9.0,Астана,Нура р-н,GreenLine.Asyl Mura Ualihanov,монолитный,свежий ремонт,unknown,3.00,4,0.333333,0,0,76,51.136749,71.397118,1
12599,1,61.00,11.0,16.0,Астана,Сарыарка р-н,KHAN RESIDENCE,монолитный,unknown,unknown,2.80,0,0.687500,0,0,88,51.194554,71.323382,1
12600,2,53.70,6.0,11.0,Шымкент,Туран р-н,unknown,монолитный,"не новый, но аккуратный ремонт",совмещенный,2.85,38,0.545455,0,0,584,42.312242,69.642768,1


In [129]:
df[features].merge(df[['price', 'price_per_m2']], on='id')

KeyError: 'id'

In [103]:
df_exp['district'].nunique()

20